# Analiz Ajanı — LoRA eğitimi (Görev 1)

Ham dilekçe veya kurum yazısını yedi alanlı JSON'a çevirir: `document_type`, `sender_type`, `primary_topic`, `requested_action`, `key_information`, `missing_information`, `summary`.

**Veri:** 2.444 kayıt — 1.957 eğitim, 246 doğrulama, 241 test. Depodaki kopyası: `analiz_qwen1/veri/*.jsonl`.

**Yöntem:** Taban model `Qwen/Qwen2.5-7B-Instruct` üzerine Unsloth ile QLoRA (4-bit) tabanlı LoRA eğitimi. Adaptör taban modele birleştirilmez; canlı demo (`app/notebooks/Agent_Demo.ipynb`) vLLM'e `analiz_lora` adıyla takar.

**Gereken:** GPU'lu bir çalışma zamanı ve `train.jsonl`, `val.jsonl`, `test.jsonl` dosyaları.

## 1) Drive'ı bağla ve çalışma klasörlerini oluştur

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive"
BASE_DIR = os.path.join(DRIVE_ROOT, "finetune_data_qwen1")
DATA_DIR = BASE_DIR
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
ADAPTER_DIR = os.path.join(DRIVE_ROOT, "lora_adapter_qwen1")
HF_CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")

for d in [BASE_DIR, OUTPUT_DIR, ADAPTER_DIR, HF_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print("DATA_DIR icerigi:", os.listdir(DATA_DIR))
assert os.path.exists(os.path.join(DATA_DIR, "train.jsonl")), "train.jsonl yok: finetune_data_qwen1 icine koy."
print("train.jsonl bulundu. LoRA kayit yolu:", ADAPTER_DIR)

## 2) Kütüphaneleri kur

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets

## 3) Taban modeli yükle

Hugging Face önbelleği Drive'a yönlendirilir; model ilk çalıştırmada indirilir, sonraki çalıştırmalarda önbellekten okunur.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096   # evraklarimiz genelde kisa/orta uzunlukta, 4096 rahat yeter
DTYPE = None             # None = otomatik (destekleyen GPU'da bfloat16 secilir)
LOAD_IN_4BIT = True

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

HF_MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

already_cached = len(os.listdir(HF_CACHE_DIR)) > 0
print(f"HF cache klasoru: {HF_CACHE_DIR}  (daha once indirilmis mi: {already_cached})")
print("Ilk calistirmada indirilecek (birkac dakika surebilir). Sonraki calistirmalarda Drive'daki cache'den okunacak, tekrar inmeyecek.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("Model hazir.")

## 4) LoRA katmanlarını ekle

Her yeni eğitimde bu hücre baştan çalıştırılır; adaptörler sıfırdan başlar.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # LoRA rank - 16 dengeli bir baslangic (bellek yeterliyse 32 da kullanilabilir)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

## 5) Veriyi yükle ve sohbet şablonuna çevir

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

train_dataset = load_dataset("json", data_files=os.path.join(DATA_DIR, "train.jsonl"), split="train")
val_dataset = load_dataset("json", data_files=os.path.join(DATA_DIR, "val.jsonl"), split="train")

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Train:", len(train_dataset), " Val:", len(val_dataset))
print("--- ornek formatlanmis metin ---")
print(train_dataset[0]["text"][:800])

## 6) Eğitimi başlat

`train_on_responses_only` ile kayıp yalnızca asistan yanıtı (üretilen JSON) üzerinden hesaplanır; sistem prompt'u ve belge metni maskelenir.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # efektif batch = 16
    warmup_ratio=0.05,
    num_train_epochs=3,                  # 856 kayit icin 3 epoch makul baslangic
    learning_rate=2e-4,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,                            # bf16 destekleyen GPU'lar icin
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

# loss'u yalnizca asistan (JSON) tokenlarindan hesapla, system+user'dan DEGIL:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()
print(trainer_stats)

## 7) Adaptörü kaydet

Adaptör taban modele birleştirilmez. Demo bu klasörü vLLM'e LoRA olarak takar.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA kaydedildi: {ADAPTER_DIR}")

## 8) Test kümesinde değerlendirme (241 kayıt)

Eğitimde kullanılmayan `test.jsonl` okunur. Model çıktıları altın etiketlerle `document_type`, `sender_type`, `primary_topic` ve eksik bilgi tespiti üzerinden karşılaştırılır.

In [ ]:
import json

FastLanguageModel.for_inference(model)  # 2x hizli inference modu

with open(os.path.join(DATA_DIR, "train.jsonl"), encoding="utf-8") as f:
    _first = json.loads(f.readline())
    SYSTEM_PROMPT_INFERENCE = _first["messages"][0]["content"]

# --- Batch inference icin gerekli ayar: sola padding (decoder-only modeller icin sart) ---
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_BATCH_SIZE = 16  # VRAM hatasi alirsan 8'e dusur

def predict_batch(document_texts: list, max_new_tokens: int = 900) -> list:
    texts = []
    for doc in document_texts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_INFERENCE},
            {"role": "user", "content": doc},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens, use_cache=True,
        do_sample=False, pad_token_id=tokenizer.pad_token_id,
    )
    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(row[input_len:], skip_special_tokens=True).strip() for row in outputs]

def load_eval_records(path):
    """Sohbet jsonl (messages) veya düz {document_text, labels} kayıtlarını okur."""
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            if "document_text" in obj and "labels" in obj:
                recs.append({"document_text": obj["document_text"], "labels": obj["labels"]})
            elif "messages" in obj:
                recs.append({
                    "document_text": obj["messages"][1]["content"],
                    "labels": json.loads(obj["messages"][2]["content"]),
                })
            else:
                raise ValueError(f"Beklenmeyen kayit bicimi: {path}")
    return recs

def first_existing(*names):
    for name in names:
        p = os.path.join(DATA_DIR, name)
        if os.path.isfile(p):
            return p
    raise FileNotFoundError("test.jsonl / test_raw.jsonl bulunamadi: " + DATA_DIR)

TEST_PATH = first_existing("test.jsonl", "test_raw.jsonl")
test_records = load_eval_records(TEST_PATH)
print("Eval dosyasi:", TEST_PATH)

print(f"Test kayit sayisi: {len(test_records)}  |  batch boyutu: {EVAL_BATCH_SIZE}")


In [ ]:
# Tam test seti - BATCH halinde uretim + metrik hesabi

import re, time

def try_parse_json(text: str):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    # Son care: cikti max_new_tokens sinirinda kesilmis olabilir (kapanmamis
    # dizi/nesne). Acik kalan { ve [ sayisina gore kapanis karakteri ekleyip
    # tekrar dene.
    candidate = text[text.find("{"):] if "{" in text else text
    # Yarim kalan bir string degeri varsa (tek sayida ") kapatmayi dene.
    if candidate.count('"') % 2 == 1:
        candidate += '"'
    opens_curly = candidate.count("{") - candidate.count("}")
    opens_square = candidate.count("[") - candidate.count("]")
    candidate = candidate.rstrip().rstrip(",")
    candidate += "]" * max(0, opens_square) + "}" * max(0, opens_curly)
    try:
        return json.loads(candidate)
    except Exception:
        return None

n = len(test_records)
json_valid = 0
doc_type_correct = 0
sender_type_correct = 0
topic_correct = 0
missing_empty_match = 0
results = []

t0 = time.time()
for i in range(0, n, EVAL_BATCH_SIZE):
    chunk = test_records[i:i + EVAL_BATCH_SIZE]
    pred_texts = predict_batch([r["document_text"] for r in chunk])
    for rec, pred_text in zip(chunk, pred_texts):
        gold = rec["labels"]
        pred = try_parse_json(pred_text)
        row = {"gold": gold, "pred_raw": pred_text, "pred_parsed": pred}
        if pred is not None:
            json_valid += 1
            if pred.get("document_type") == gold.get("document_type"):
                doc_type_correct += 1
            if pred.get("sender_type") == gold.get("sender_type"):
                sender_type_correct += 1
            if pred.get("primary_topic") == gold.get("primary_topic"):
                topic_correct += 1
            gold_mi_empty = (gold.get("missing_information") == [])
            pred_mi_empty = (pred.get("missing_information") in ([], None))
            if gold_mi_empty == pred_mi_empty:
                missing_empty_match += 1
        results.append(row)
    print(f"  {min(i+EVAL_BATCH_SIZE, n)}/{n} tamamlandi ({time.time()-t0:.0f}sn gecti)")

print(f"\nToplam sure: {time.time()-t0:.0f} saniye")
print(f"Toplam test kaydi: {n}")
print(f"Gecerli JSON orani: {json_valid}/{n} (%{100*json_valid/n:.1f})")
print(f"document_type dogruluk: {doc_type_correct}/{n} (%{100*doc_type_correct/n:.1f})")
print(f"sender_type dogruluk: {sender_type_correct}/{n} (%{100*sender_type_correct/n:.1f})")
print(f"primary_topic dogruluk: {topic_correct}/{n} (%{100*topic_correct/n:.1f})")
print(f"missing_information (bos/dolu) dogruluk: {missing_empty_match}/{n} (%{100*missing_empty_match/n:.1f})")

with open(os.path.join(BASE_DIR, "test_eval_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Detayli sonuclar kaydedildi: {os.path.join(BASE_DIR, 'test_eval_results.json')}")

In [ ]:
# Taksonomi disi ("hallucinated") etiket kontrolu + en yakin gecerli etikete yakalama
#
# Model bazen primary_topic/document_type icin taksonomide (58 primary_topic,
# 5 document_type) olmayan, iki kategoriyi harmanlayan yeni bir isim uretebiliyor
# (ornek: "SALDIRGAN_RISKLI_HAYVAN_KIMININ_DENETIMI", "OTOPARK_DUZENLEME").
# Bu hucre once bunu tespit eder, sonra difflib ile en yakin GECERLI etikete
# "yakalayarak" (post-hoc duzeltme) duzeltilmis dogruluk oranini gosterir.
#
# NOT: Bu, gercek constrained decoding'in YERINE GECMEZ -- sadece uretimden
# SONRA yapilan bir guvenlik agidir ve modelin uretmedigi dogru etiketi geri
# getiremez. Gercek cozum: sunuma gecerken (vLLM / outlines / lm-format-enforcer
# gibi guided-decoding destekleyen bir stack ile) primary_topic ve document_type
# alanlarini JSON-schema/enum'a kilitlemek -- boylece taksonomi disi bir deger
# uretmek YAPISAL OLARAK imkansiz hale gelir. transformers.generate + Unsloth ile
# calisiyorsan, alternatif olarak bir LogitsProcessor yazip yalnizca gecerli enum
# degerlerine karsilik gelen token dizilerine izin verebilirsin.

import difflib
from collections import Counter

def _labels_of(rec):
    if "labels" in rec:
        return rec["labels"]
    return json.loads(rec["messages"][2]["content"])

def load_allowed_labels(field, *raw_paths):
    allowed = set()
    for p in raw_paths:
        if not os.path.isfile(p):
            continue
        with open(p, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                allowed.add(_labels_of(rec)[field])
    if not allowed:
        raise FileNotFoundError("Etiket dosyasi yok: " + ", ".join(raw_paths))
    return allowed

ALLOWED_TOPICS = load_allowed_labels(
    "primary_topic",
    os.path.join(DATA_DIR, "train.jsonl"),
    os.path.join(DATA_DIR, "val.jsonl"),
    os.path.join(DATA_DIR, "test.jsonl"),
    os.path.join(DATA_DIR, "train_raw.jsonl"),
    os.path.join(DATA_DIR, "val_raw.jsonl"),
    os.path.join(DATA_DIR, "test_raw.jsonl"),
)
ALLOWED_DOC_TYPES = load_allowed_labels(
    "document_type",
    os.path.join(DATA_DIR, "train.jsonl"),
    os.path.join(DATA_DIR, "val.jsonl"),
    os.path.join(DATA_DIR, "test.jsonl"),
    os.path.join(DATA_DIR, "train_raw.jsonl"),
    os.path.join(DATA_DIR, "val_raw.jsonl"),
    os.path.join(DATA_DIR, "test_raw.jsonl"),
)

print(f"Taksonomideki primary_topic sayisi: {len(ALLOWED_TOPICS)}")
print(f"Taksonomideki document_type sayisi: {len(ALLOWED_DOC_TYPES)}")

off_taxonomy = Counter()
topic_correct_corrected = 0
doc_type_correct_corrected = 0
n_scored = 0

for row in results:
    pred = row["pred_parsed"]
    gold = row["gold"]
    if pred is None:
        continue
    n_scored += 1

    pred_topic = pred.get("primary_topic")
    if pred_topic not in ALLOWED_TOPICS:
        off_taxonomy[("primary_topic", pred_topic)] += 1
        match = difflib.get_close_matches(pred_topic or "", ALLOWED_TOPICS, n=1, cutoff=0.0)
        pred_topic = match[0] if match else pred_topic
    if pred_topic == gold.get("primary_topic"):
        topic_correct_corrected += 1

    pred_doc = pred.get("document_type")
    if pred_doc not in ALLOWED_DOC_TYPES:
        off_taxonomy[("document_type", pred_doc)] += 1
        match = difflib.get_close_matches(pred_doc or "", ALLOWED_DOC_TYPES, n=1, cutoff=0.0)
        pred_doc = match[0] if match else pred_doc
    if pred_doc == gold.get("document_type"):
        doc_type_correct_corrected += 1

print()
print(f"Taksonomi disi tahmin sayisi (JSON gecerli olan {n_scored} kayit icinde):")
if off_taxonomy:
    for (field, val), cnt in off_taxonomy.most_common():
        print(f"  {cnt}x  {field} = {val!r}")
else:
    print("  yok")

print()
print(f"primary_topic dogruluk (duzeltme ONCESI): {topic_correct}/{n} (%{100*topic_correct/n:.1f})")
print(f"primary_topic dogruluk (en-yakin-etikete duzeltme SONRASI): {topic_correct_corrected}/{n} (%{100*topic_correct_corrected/n:.1f})")
print(f"document_type dogruluk (duzeltme ONCESI): {doc_type_correct}/{n} (%{100*doc_type_correct/n:.1f})")
print(f"document_type dogruluk (en-yakin-etikete duzeltme SONRASI): {doc_type_correct_corrected}/{n} (%{100*doc_type_correct_corrected/n:.1f})")


Canlı çıkarım bu notebook'ta yapılmaz. Demo: `app/notebooks/Agent_Demo.ipynb` — vLLM üzerinde Instruct taban + bu adaptör (`analiz_lora`).